# 3. Modelos base

La tercera pestaña del dashboard presenta el modelo base, su comparación con dos referencias,
la verificación del desempeño, las curvas de evaluación y el diagnóstico de sobreajuste. El
dashboard muestra resultados precalculados, puesto que el entrenamiento se realiza una sola vez
fuera de la aplicación; este capítulo ejecuta ese entrenamiento a partir de la matriz de
predictores construida en el capítulo 2, por lo que requiere haber ejecutado antes ese
notebook.

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import scipy.stats as st
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve)

warnings.filterwarnings("ignore")

df = pd.read_pickle("datos/features.pkl")
par = json.load(open("datos/predictores.json", encoding="utf-8"))
LIM = tuple(par["LIM"])
BASE_COLS, DOM_COLS = par["BASE_COLS"], par["DOM_COLS"]
TODAS = BASE_COLS + DOM_COLS

CORTE = pd.Timestamp("2010-01-01")
INICIO_TEST = CORTE + pd.Timedelta(days=30)

dm = df.dropna(subset=["y"])
TR = dm[dm.fecha < CORTE]
TE = dm[dm.fecha >= INICIO_TEST]
ytr, yte = TR.y.astype(int), TE.y.astype(int)

print(f"Entrenamiento: {len(TR):,} días, prevalencia {ytr.mean():.1%}")
print(f"Prueba:        {len(TE):,} días, prevalencia {yte.mean():.1%}")

Entrenamiento: 25,364 días, prevalencia 25.7%
Prueba:        6,066 días, prevalencia 24.9%


## 3.1 Modelo base y referencias

El modelo base es una regresión logística con ponderación equilibrada de clases, que asigna
a cada observación un peso inversamente proporcional a la frecuencia de su clase y compensa
así el desbalance de 2,9 a 1. La regresión se integra en un Pipeline con imputación por
mediana y escalado estándar, de modo que ambas transformaciones se ajustan exclusivamente con
el conjunto de entrenamiento y se aplican sin modificación al conjunto de prueba (Pedregosa
et al., 2011).

El desempeño se compara con dos referencias. La primera es un clasificador trivial que
predice siempre la clase mayoritaria y que fija el accuracy alcanzable sin información
alguna. La segunda es la persistencia, que supone que dentro de quince días el río se
encontrará en la misma condición que hoy y que, dada la autocorrelación de 0,918 a ese
horizonte, constituye la referencia exigente propia del dominio hidrológico.

In [2]:
def modelo_base():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("esc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)),
    ])


modelo = modelo_base().fit(TR[TODAS], ytr)
ys = modelo.predict_proba(TE[TODAS])[:, 1]
yp = modelo.predict(TE[TODAS])

dummy = DummyClassifier(strategy="most_frequent").fit(TR[TODAS], ytr)
yp_dum = dummy.predict(TE[TODAS])
yp_per = TE["calamar"].between(*LIM).astype(int).values

filas = []
for nombre, pred, score in [("Dummy", yp_dum, yp_dum.astype(float)),
                            ("Persistencia", yp_per, yp_per.astype(float)),
                            ("Regresión logística", yp, ys)]:
    filas.append({"modelo": nombre,
                  "accuracy": accuracy_score(yte, pred),
                  "precision": precision_score(yte, pred, zero_division=0),
                  "recall": recall_score(yte, pred, zero_division=0),
                  "f1": f1_score(yte, pred, zero_division=0),
                  "roc_auc": roc_auc_score(yte, score),
                  "pr_auc": average_precision_score(yte, score)})
metricas = pd.DataFrame(filas).round(3)
metricas

,modelo,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Dummy,0.751,0.000,0.000,0.000,0.500,0.249
1,Persistencia,0.883,0.764,0.764,0.764,0.843,0.642
2,Regresión logística,0.882,0.695,0.937,0.798,0.959,0.872


En el periodo de prueba, que se inicia el 31 de enero de 2010 y llega a septiembre de 2026, el
clasificador trivial alcanza un accuracy de 0,751 con exhaustividad nula, puesto que nunca
predice la clase positiva. La persistencia obtiene un accuracy de 0,883 y un F1 de 0,764, y su
precisión coincide con su exhaustividad porque la proporción de días en la banda resulta
prácticamente igual en $t$ y en $t+15$. La regresión logística alcanza un accuracy de 0,882,
equivalente al de la persistencia, aunque supera a ambas referencias en F1, con 0,798, en
área bajo la curva ROC, con 0,959, y en área bajo la curva precisión-exhaustividad, con 0,872
frente a 0,642 de la persistencia.

Las referencias de la persistencia y del clasificador trivial carecen de una puntuación
continua, de modo que sus áreas bajo las curvas se calculan con la predicción binaria y
equivalen a un único punto de operación. Esa diferencia no altera la comparación, dado que el
modelo base también supera a la persistencia en F1, métrica que se calcula sobre la predicción
binaria de ambos.

## 3.2 Verificación del desempeño

Un accuracy de 0,882 exige descartar tres explicaciones alternativas antes de atribuirlo a la
capacidad del modelo, a saber, una fuga de datos, un problema trivial o un efecto del
desbalance de clases. La fuga se descartó mediante la auditoría de la sección 2.7. El
desbalance explica que un clasificador sin información alcance 0,751. La trivialidad del
problema se examina variando el horizonte de predicción, puesto que un problema trivial
mantendría el desempeño elevado con independencia de la antelación exigida.

In [3]:
filas = []
for h in (1, 7, 15, 30, 60):
    z = df.copy()
    z["y"] = z["calamar"].between(*LIM).astype(float).shift(-h)
    z = z.dropna(subset=["y"])
    a, b = z[z.fecha < CORTE], z[z.fecha >= INICIO_TEST]
    yb = b.y.astype(int)
    m = modelo_base().fit(a[TODAS], a.y.astype(int))
    persist = b["calamar"].between(*LIM).astype(int)
    filas.append({"Horizonte (días)": h,
                  "Accuracy modelo": round(accuracy_score(yb, m.predict(b[TODAS])), 3),
                  "Accuracy persistencia": round(accuracy_score(yb, persist), 3),
                  "Accuracy dummy": round(1 - yb.mean(), 3),
                  "PR-AUC modelo": round(average_precision_score(yb, m.predict_proba(b[TODAS])[:, 1]), 3),
                  "PR-AUC persistencia": round(average_precision_score(yb, persist), 3)})
verificacion = pd.DataFrame(filas)
verificacion

,Horizonte (días),Accuracy modelo,Accuracy persistencia,Accuracy dummy,PR-AUC modelo,PR-AUC persistencia
0,1,0.982,0.989,0.752,0.996,0.962
1,7,0.932,0.939,0.752,0.955,0.801
2,15,0.882,0.883,0.751,0.872,0.642
3,30,0.766,0.790,0.751,0.629,0.439
4,60,0.701,0.657,0.750,0.483,0.271


A un día de antelación el modelo y la persistencia superan un accuracy de 0,98, lo que
confirma que a corto plazo el problema se resuelve repitiendo el estado actual. A medida que
el horizonte crece ambos descienden, y a sesenta días el accuracy del modelo, 0,701, queda por
debajo del que obtiene el clasificador trivial, 0,750, efecto de la ponderación equilibrada,
que sacrifica aciertos en la clase mayoritaria para recuperar la minoritaria. El problema, en
consecuencia, se vuelve difícil con la antelación y el accuracy de 0,882 a quince días refleja
principalmente la persistencia del caudal.

El PR-AUC distingue con mayor claridad la contribución del modelo. La ventaja sobre la
persistencia crece de 0,034 a un día a 0,154 a siete y alcanza su máximo a quince días, con
0,230, antes de reducirse a 0,190 a treinta días y a 0,212 a sesenta. El horizonte de quince
días coincide, por lo tanto, con el punto en que el modelo aporta la mayor información
adicional respecto de la persistencia.

## 3.3 Curvas de evaluación

Las curvas se calculan con la probabilidad predicha por el modelo base sobre el conjunto de
prueba. Con una clase positiva del 24,9 %, la curva precisión-exhaustividad resulta más
informativa que la ROC, porque su línea de referencia depende de la prevalencia y su forma
refleja directamente el costo de las falsas alarmas sobre la clase minoritaria (Saito y
Rehmsmeier, 2015).

In [4]:
prec, rec, _ = precision_recall_curve(yte, ys)
fig = px.line(x=rec, y=prec, labels={"x": "Exhaustividad", "y": "Precisión"})
fig.add_hline(y=yte.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"azar ({yte.mean():.2f})")
fig.update_layout(height=400, margin=dict(t=10))
fig

La precisión se aproxima a 0,9 con una exhaustividad de 0,5 y se mantiene por encima de 0,7
hasta una exhaustividad de 0,9, de modo que el modelo recupera la mayor parte de los días de
transporte dominante con una proporción moderada de falsas alarmas. Más allá de ese punto la
curva desciende hacia la línea de referencia, de 0,25, que corresponde a una predicción
aleatoria. El área bajo la curva, 0,872, equivale a 3,5 veces ese valor de referencia.

In [5]:
fpr, tpr, _ = roc_curve(yte, ys)
fig = px.line(x=fpr, y=tpr, labels={"x": "Tasa de falsos positivos",
                                    "y": "Tasa de verdaderos positivos"})
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(height=400, margin=dict(t=10))
fig

El área bajo la curva ROC alcanza 0,959, frente a 0,5 de un clasificador sin
información, representado por la diagonal. La curva asciende con rapidez en la región de
tasas de falsos positivos bajas, lo que indica que el modelo ordena correctamente la mayor
parte de los días positivos por encima de los negativos.

In [6]:
cm = confusion_matrix(yte, yp)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                x=["predicho: no", "predicho: sí"], y=["real: no", "real: sí"])
fig.update_layout(height=400, margin=dict(t=10), coloraxis_showscale=False)
fig

Con el umbral de 0,5, el modelo identifica 1.413 de los 1.508 días de transporte
dominante del periodo de prueba, una exhaustividad de 0,937, a costa de 621 falsas alarmas
que reducen la precisión a 0,695. El balance responde a la ponderación equilibrada de clases,
que penaliza con mayor peso los días positivos no detectados. Para la operación de embalses o
la programación de dragados, una falsa alarma implica preparar una intervención que no se
requiere, mientras que un día no detectado implica no anticipar un periodo de transporte
dominante, de manera que el umbral podría ajustarse según el costo relativo de ambos errores
en cada aplicación.

In [7]:
d = verificacion.melt(id_vars="Horizonte (días)",
                      value_vars=["PR-AUC modelo", "PR-AUC persistencia"],
                      var_name="serie", value_name="PR-AUC")
fig = px.line(d, x="Horizonte (días)", y="PR-AUC", color="serie", markers=True,
              labels={"serie": ""})
fig.update_layout(height=400, margin=dict(t=10))
fig

La figura representa la columna de PR-AUC de la tabla de verificación. A un día ambas
curvas se sitúan cerca de 1, y a medida que el horizonte crece las dos descienden, aunque la
del modelo lo hace con mayor lentitud, de modo que la separación entre ambas se amplía hasta
los quince días y se mantiene a partir de ese punto.

## 3.4 Sobreajuste y limitaciones

El sobreajuste se evaluó mediante validación cruzada temporal de cinco pliegues sobre el
periodo de entrenamiento, con un intervalo de treinta días entre el final de cada bloque de
ajuste y el inicio del bloque de validación. Para cada pliegue se registró el área bajo la
curva ROC en ambos bloques, y la diferencia se contrastó mediante una prueba t pareada.

In [8]:
cv = TimeSeriesSplit(n_splits=5, gap=30)
r = cross_validate(modelo_base(), TR[TODAS], ytr, cv=cv, scoring="roc_auc",
                   return_train_score=True)

print(pd.DataFrame({"pliegue": range(1, 6),
                    "ROC-AUC entrenamiento": r["train_score"],
                    "ROC-AUC validación": r["test_score"]}).round(4).to_string(index=False))

dif = r["train_score"] - r["test_score"]
t, p = st.ttest_rel(r["train_score"], r["test_score"])
print(f"\nDiferencia media: {dif.mean():+.3f}   t = {t:.3f}   p = {p:.3f}")

 pliegue  ROC-AUC entrenamiento  ROC-AUC validación
       1                 0.9835              0.9191
       2                 0.9669              0.9077
       3                 0.9601              0.8949
       4                 0.9608              0.9659
       5                 0.9626              0.9360

Diferencia media: +0.042   t = 3.055   p = 0.038


La diferencia media entre entrenamiento y validación es de 0,042 en ROC-AUC, y la prueba
t pareada la declara significativa (t = 3,055, p = 0,038). La significación estadística
indica que la diferencia es sistemática a lo largo de los pliegues, aunque su magnitud, de
cuatro puntos porcentuales, corresponde a un sobreajuste leve, coherente con un modelo lineal
regularizado de 43 coeficientes ajustado sobre varias decenas de miles de días. Con solo
cinco pliegues, la prueba dispone de cuatro grados de libertad, por lo que el valor p debe
leerse con cautela.

La limitación principal del modelo base es de forma funcional. El objetivo es una banda de
caudales, condición no monótona en la que tanto los caudales demasiado bajos como los
demasiado altos pertenecen a la clase negativa, y una frontera lineal sobre el caudal no puede
representarla. Las variables de dominio, que miden la distancia del caudal a la descarga
efectiva, transforman la banda en una relación monótona. La comparación siguiente ajusta el
modelo con y sin esas variables en cada horizonte.

In [9]:
filas = []
for h in (1, 7, 15, 30, 60):
    z = df.copy()
    z["y"] = z["calamar"].between(*LIM).astype(float).shift(-h)
    z = z.dropna(subset=["y"])
    a, b = z[z.fecha < CORTE], z[z.fecha >= INICIO_TEST]
    fila = {"Horizonte (días)": h}
    for nombre, cols in [("Solo base", BASE_COLS), ("Base y dominio", TODAS)]:
        m = modelo_base().fit(a[cols], a.y.astype(int))
        fila[nombre] = round(average_precision_score(b.y.astype(int),
                                                     m.predict_proba(b[cols])[:, 1]), 3)
    filas.append(fila)
pd.DataFrame(filas)

,Horizonte (días),Solo base,Base y dominio
0,1,0.481,0.996
1,7,0.483,0.955
2,15,0.487,0.872
3,30,0.476,0.629
4,60,0.459,0.483


Sin las variables de dominio, el modelo lineal se estanca en un PR-AUC cercano a 0,48 en
todos los horizontes, incluido el de un día, en el que la persistencia supera 0,96. Ese
resultado confirma que la capacidad del modelo base depende de una transformación construida
con conocimiento del problema y que, por sí sola, la regresión logística no logra representar
la pertenencia a una banda. La limitación motiva la evaluación de clasificadores capaces de
aprender fronteras no monótonas directamente a partir de los predictores base.